# PCGT on ogbn-arxiv (Large-Scale Experiment)

Run PCGT on ogbn-arxiv (~170K nodes) on Colab/Kaggle GPU.
SGFormer baseline: **72.63 ± 0.13**

**Runtime**: GPU recommended (T4/V100). Full-graph training, ~170K nodes fit in GPU memory.

## 1. Setup

In [1]:
# Mount Google Drive (Colab) or set paths (Kaggle)
import os
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/PCGT'
    DATA_DIR = '/content/PCGT/data/'
elif IN_KAGGLE:
    REPO_ROOT = '/kaggle/working/PCGT'
    DATA_DIR = '/kaggle/working/PCGT/data/'
else:
    REPO_ROOT = '.'
    DATA_DIR = '../data/'

print(f'Platform: {"Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"}')
print(f'REPO_ROOT: {REPO_ROOT}')

Mounted at /content/drive
Platform: Colab
REPO_ROOT: /content/PCGT


In [3]:
# Clone repo (skip if already exists), always pull latest
if not os.path.exists(f'{REPO_ROOT}/large/main.py'):
    !git clone https://github.com/ranjanchoubey/PCGT.git {REPO_ROOT}
else:
    print(f'Repo already at {REPO_ROOT}, pulling latest...')
    !cd {REPO_ROOT} && git pull origin dev

# If on Colab and repo is on Drive, symlink data
if IN_COLAB and os.path.exists('/content/drive/MyDrive/PCGT/data'):
    !ln -sf /content/drive/MyDrive/PCGT/data {REPO_ROOT}/data
    print('Symlinked data from Drive')

Repo already at /content/PCGT, pulling latest...
From https://github.com/ranjanchoubey/PCGT
 * branch            dev        -> FETCH_HEAD
Updating a2ded85..d2f22bb
Fast-forward
 .gitignore                                         |    9 +-
 FINAL_CONFIGS.sh                                   |   42 +-
 GloGNN.pdf                                         |  Bin 0 -> 1129828 bytes
 PCGT_Runner.ipynb                                  | 1174 +++++++++-----------
 PCGT_arxiv.ipynb                                   |  601 ++++++++++
 large/data_utils.py                                |    5 +-
 large/dataset.py                                   |   14 +-
 large/main.py                                      |    8 +
 large/pcgt.py                                      |  259 +++++
 medium/_archive/pcgt_v5_ffn.py                     |  328 ++++++
 .../ablation_log.txt                               |  207 ++++
 .../chameleon_01_full_pcgt.txt                     |   65 ++
 .../chameleon_02_local_only.

In [8]:
import torch

# Uninstall previous versions to prevent conflicts
!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster --y

# Install compatible versions using the PyG index URL
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Install PyTorch Geometric
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

# Install remaining dependencies
!pip install ogb pymetis googledrivedownloader performer_pytorch

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 118.1 MB/s eta 0:00:0000:0100:01
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 81.4 MB/s eta 0:00:00a 0:00:01
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.6 MB/s eta 0:00:00a 0:00:01
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-0e22mvm3
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-0e22mvm3
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit f1e6765616f96d9fc5e222c7e18c55d36b497057
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for torch-geometric: filename=torch_geometric

In [4]:
# Verify
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128, CUDA: True
GPU: Tesla T4


In [9]:
!nvidia-smi

Mon Mar 23 11:23:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
os.chdir(f'{REPO_ROOT}/large')
print(f'Working directory: {os.getcwd()}')
!ls *.py

Working directory: /content/PCGT/large
dataset.py     eval.py	load_data.py  main-batch.py  ours.py   partition.py
data_utils.py  gnns.py	logger.py     main.py	     parse.py  pcgt.py


In [6]:
# Fix for PyTorch 2.6+ — torch.load defaults to weights_only=True
# which breaks OGB's pickle-based dataset caching.
# Safe to re-run: stores original on torch module to prevent recursion.
import torch
if not hasattr(torch, '_pcgt_original_load'):
    torch._pcgt_original_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return torch._pcgt_original_load(*args, **kwargs)
    torch.load = _patched_torch_load
    print('Patched torch.load for OGB compatibility')
else:
    print('torch.load already patched (safe re-run)')

Patched torch.load for OGB compatibility


In [ ]:
# Create log directory next to this notebook
LOG_DIR = f'{REPO_ROOT}/logs'
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Logs will be saved to: {LOG_DIR}')

## 2. Download ogbn-arxiv
OGB auto-downloads to `data/ogb/`. ~170K nodes, 128 features, 40 classes.

In [ ]:
# Pre-download dataset so we can inspect it
from ogb.nodeproppred import NodePropPredDataset
dataset = NodePropPredDataset(name='ogbn-arxiv', root=f'{DATA_DIR}ogb')
data = dataset[0]
print(f'Nodes: {data[0]["num_nodes"]:,}')
print(f'Edges: {data[0]["edge_index"].shape[1]:,}')
print(f'Features: {data[0]["node_feat"].shape[1]}')
print(f'Classes: {dataset.num_classes}')
split = dataset.get_idx_split()
print(f'Train: {len(split["train"]):,}, Val: {len(split["valid"]):,}, Test: {len(split["test"]):,}')

## 3. SGFormer Baseline
Reproduce the paper number: **72.63 ± 0.13** (5 runs)

In [ ]:
# SGFormer baseline — 5 runs (matches run.sh exactly)
!python main.py --method sgformer --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --seed 123 --runs 5 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR} 2>&1 | tee {LOG_DIR}/sgformer_baseline.log

Namespace(dataset='ogbn-arxiv', sub_dataset='', data_dir='/content/PCGT/data/', device=0, seed=123, cpu=False, epochs=1000, runs=5, directed=False, train_prop=0.5, valid_prop=0.25, protocol='semi', rand_split=False, rand_split_class=False, label_num_per_class=20, metric='acc', method='sgformer', hidden_channels=256, use_graph=True, aggregate='add', graph_weight=0.5, gnn_use_bn=True, gnn_use_residual=True, gnn_use_weight=True, gnn_use_init=False, gnn_use_act=True, gnn_num_layers=3, gnn_dropout=0.5, gnn_weight_decay=0.0, trans_num_heads=1, trans_use_weight=True, trans_use_bn=True, trans_use_residual=True, trans_use_act=False, trans_num_layers=1, trans_dropout=0.5, trans_weight_decay=0.0, lr=0.001, batch_size=10000, patience=200, display_step=1, eval_step=9, cached=False, print_prop=False, save_result=False, save_model=False, use_pretrained=False, save_att=False, model_dir='../../model/', hops=2, gat_heads=4, out_heads=1, num_partitions=500, partition_method='metis')
ogbn-arxiv
dataset og

## 4. PCGT on ogbn-arxiv

Start with SGFormer's config, add partition attention.
Key parameters to tune: `--num_partitions`, `--graph_weight`, `--partition_method`

In [ ]:
# PCGT Config A1: METIS K=256, graph_weight=0.5 (balanced, same as SGFormer)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 256 --partition_method metis \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR} 2>&1 | tee {LOG_DIR}/pcgt_a1_k256_gw05.log

Namespace(dataset='ogbn-arxiv', sub_dataset='', data_dir='/content/PCGT/data/', device=0, seed=123, cpu=False, epochs=1000, runs=3, directed=False, train_prop=0.5, valid_prop=0.25, protocol='semi', rand_split=False, rand_split_class=False, label_num_per_class=20, metric='acc', method='pcgt', hidden_channels=256, use_graph=True, aggregate='add', graph_weight=0.5, gnn_use_bn=True, gnn_use_residual=True, gnn_use_weight=True, gnn_use_init=False, gnn_use_act=True, gnn_num_layers=3, gnn_dropout=0.5, gnn_weight_decay=0.0, trans_num_heads=1, trans_use_weight=True, trans_use_bn=True, trans_use_residual=True, trans_use_act=False, trans_num_layers=1, trans_dropout=0.5, trans_weight_decay=0.0, lr=0.001, batch_size=10000, patience=200, display_step=1, eval_step=9, cached=False, print_prop=False, save_result=False, save_model=False, use_pretrained=False, save_att=False, model_dir='../../model/', hops=2, gat_heads=4, out_heads=1, num_partitions=256, partition_method='metis')
ogbn-arxiv
Downloaded 0.0

In [ ]:
# PCGT Config A2: METIS K=500 (finer partitions, smaller local attention)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 500 --partition_method metis \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR} 2>&1 | tee {LOG_DIR}/pcgt_a2_k500_gw05.log

/content/PCGT/large/logger.py:76: SyntaxWarning: invalid escape sequence '\p'
  f"{results.mean():.2f} $\pm$ {results.std():.2f} \n")
Namespace(dataset='ogbn-arxiv', sub_dataset='', data_dir='/content/PCGT/data/', device=0, seed=123, cpu=False, epochs=1000, runs=3, directed=False, train_prop=0.5, valid_prop=0.25, protocol='semi', rand_split=False, rand_split_class=False, label_num_per_class=20, metric='acc', method='pcgt', hidden_channels=256, use_graph=True, aggregate='add', graph_weight=0.5, gnn_use_bn=True, gnn_use_residual=True, gnn_use_weight=True, gnn_use_init=False, gnn_use_act=True, gnn_num_layers=3, gnn_dropout=0.5, gnn_weight_decay=0.0, trans_num_heads=1, trans_use_weight=True, trans_use_bn=True, trans_use_residual=True, trans_use_act=False, trans_num_layers=1, trans_dropout=0.5, trans_weight_decay=0.0, lr=0.001, batch_size=10000, patience=200, display_step=1, eval_step=9, cached=False, print_prop=False, save_result=False, save_model=False, use_pretrained=False, save_att=Fals

In [ ]:
# PCGT Config A3: METIS K=256, graph_weight=0.8 (more GCN, like medium configs)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.8 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 256 --partition_method metis \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR} 2>&1 | tee {LOG_DIR}/pcgt_a3_k256_gw08.log

In [ ]:
# PCGT Config A4: Random partitions K=256 (ablation)
!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight 0.5 \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions 256 --partition_method random \
    --seed 123 --runs 3 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR} 2>&1 | tee {LOG_DIR}/pcgt_a4_random_k256.log

## 5. Validation: Best Config × 5 Runs
Once we find the best config above, run 5 runs for the paper.

In [ ]:
# Fill in best config from above and run 5 times
# PCGT 5-run validation (EDIT: fill in best K and graph_weight)
BEST_K = 256          # <-- update after probes
BEST_GW = 0.5         # <-- update after probes
BEST_METHOD = 'metis'  # <-- update after probes

!python main.py --method pcgt --dataset ogbn-arxiv --metric acc \
    --lr 0.001 --hidden_channels 256 \
    --use_graph --graph_weight {BEST_GW} \
    --gnn_num_layers 3 --gnn_dropout 0.5 --gnn_weight_decay 0. \
    --gnn_use_residual --gnn_use_weight --gnn_use_bn --gnn_use_act \
    --trans_num_layers 1 --trans_dropout 0.5 --trans_weight_decay 0. \
    --trans_use_residual --trans_use_weight --trans_use_bn \
    --num_partitions {BEST_K} --partition_method {BEST_METHOD} \
    --seed 123 --runs 5 --epochs 1000 --eval_step 9 --device 0 \
    --data_dir {DATA_DIR} 2>&1 | tee {LOG_DIR}/pcgt_best_5runs.log

## 6. Results Summary

In [ ]:
print('='*60)
print('OGBN-ARXIV RESULTS SUMMARY')
print('='*60)
print(f'SGFormer (paper):  72.63 ± 0.13')
print(f'SGFormer (ours):   [fill after cell 7]')
print(f'PCGT A1 (K=256):   [fill after cell 9]')
print(f'PCGT A2 (K=500):   [fill after cell 10]')
print(f'PCGT A3 (gw=0.8):  [fill after cell 11]')
print(f'PCGT A4 (random):  [fill after cell 12]')
print(f'PCGT best (5-run): [fill after cell 14]')
print('='*60)